# Graph RAG

Plant → systems/components (graph traversal) or free-text question (embed → expand → Nova).

In [1]:
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

CHECKPOINT_PATH = ROOT / "requirements_checkpoint_embed_tier.json"
DOC_ID = "train_jsonl"

In [2]:
from src.schema import Entry

data = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
requirements_with_tiers = [(Entry(**d["entry"]), d["tier"]) for d in data["requirements_with_tiers"]]
embeddings = data.get("embeddings") or []
DOC_ID = data.get("doc_id", DOC_ID)

if len(embeddings) != len(requirements_with_tiers):
    raise ValueError("Checkpoint must have embeddings. Run LETSGO_embed_tier_CLEAN first.")

from src.neo4j_loader import requirement_full_id
full_ids = [requirement_full_id(DOC_ID, e, i) for i, (e, _) in enumerate(requirements_with_tiers)]
print(f"Loaded {len(requirements_with_tiers)} requirements, {len(full_ids)} ids.")

Loaded 2000 requirements, 2000 ids.


In [3]:
from src.neo4j_loader import get_driver

driver = get_driver()
with driver.session() as session:
    r = session.run("MATCH (n:Requirement) RETURN count(n) AS c").single()
    e = session.run("MATCH ()-[r:TRACES_TO]->() RETURN count(r) AS c").single()
driver.close()
print(f"Neo4j: {r['c']} Requirement nodes, {e['c']} TRACES_TO edges.")

Neo4j: 2000 Requirement nodes, 3080 TRACES_TO edges.


## 1. Query by plant

In [13]:
from src.neo4j_loader import requirement_full_id

plant_indices = [i for i, (_, t) in enumerate(requirements_with_tiers) if (t or "").strip().lower() == "plant"]
plant_full_ids = [full_ids[i] for i in plant_indices]
print(f"Found {len(plant_full_ids)} plant-level requirements (indices 0 to {len(plant_full_ids)-1}).")

for idx in range(min(5, len(plant_indices))):
    i = plant_indices[idx]
    e, _ = requirements_with_tiers[i]
    print(f"  [{idx}] {e.text[:80]}...")

PLANT_INDEX = 0
if PLANT_INDEX < len(plant_full_ids):
    chosen_plant_id = plant_full_ids[PLANT_INDEX]
    print(f"\nChosen plant id: {chosen_plant_id}")
else:
    chosen_plant_id = None
    print("Set PLANT_INDEX to a valid index (0 to", len(plant_full_ids) - 1, ")")

Found 460 plant-level requirements (indices 0 to 459).
  [0] Holtec's valuable technical background intellectual property is embodied in the ...
  [1] The Holtec SMR-300 is comprised of a number of systems that work together to pro...
  [2] Holtec's valuable technical background intellectual property is embodied in the ...
  [3] Holtec's valuable technical background intellectual property is embodied in the ...
  [4] Holtec's valuable technical background intellectual property is embodied in the ...

Chosen plant id: train_jsonl_048d83a13be99390_1685


In [14]:
def get_requirements_linked_to_plant(plant_id, driver, include_components=True):
    with driver.session() as session:
        if include_components:
            result = session.run(
                """
                MATCH (plant:Requirement {id: $plant_id})
                OPTIONAL MATCH (plant)-[:TRACES_TO]-(sys:Requirement)
                OPTIONAL MATCH (sys)-[:TRACES_TO]-(comp:Requirement)
                WITH collect(DISTINCT plant) + collect(DISTINCT sys) + collect(DISTINCT comp) AS nodes
                UNWIND nodes AS n
                WITH n WHERE n IS NOT NULL
                WITH DISTINCT n
                RETURN n.id AS id, n.tier AS tier, n.text AS text
                """,
                plant_id=plant_id,
            )
        else:
            result = session.run(
                """
                MATCH (plant:Requirement {id: $plant_id})-[:TRACES_TO]-(other:Requirement)
                WHERE other.tier = 'system'
                RETURN plant.id AS plant_id, plant.tier AS plant_tier, plant.text AS plant_text,
                       other.id AS id, other.tier AS tier, other.text AS text
                """,
                plant_id=plant_id,
            )
            rows = list(result)
            plant_node = (rows[0]["plant_id"], rows[0]["plant_tier"] or "", (rows[0]["plant_text"] or "")[:500]) if rows else None
            systems = [(r["id"], r["tier"] or "", (r["text"] or "")[:500]) for r in rows]
            return (plant_node, systems, [])
        nodes = [(r["id"], r["tier"] or "", (r["text"] or "")[:500]) for r in result]
    plant_node = next((n for n in nodes if n[1] == "plant"), None)
    systems = [n for n in nodes if n[1] == "system"]
    components = [n for n in nodes if n[1] == "component"] if include_components else []
    return (plant_node, systems, components)

In [15]:
from src.neo4j_loader import get_driver

driver = get_driver()
plant_node, systems, components = get_requirements_linked_to_plant(
    chosen_plant_id, driver, include_components=True
)
driver.close()

print(f"Plant: 1 requirement")
print(f"Systems linked to this plant: {len(systems)}")
print(f"Components (under those systems): {len(components)}")
if plant_node:
    print(f"\n[plant] {plant_node[2][:200]}...")
print("\n--- System-level requirements ---")
for i, (sid, tier, text) in enumerate(systems[:15]):
    print(f"  {i+1}. [{tier}] {text[:120]}...")
if len(systems) > 15:
    print(f"  ... and {len(systems) - 15} more")

Plant: 1 requirement
Systems linked to this plant: 4
Components (under those systems): 1

[plant] - With the exception of items/services that go through a dedication process, procurement documents shall require suppliers to have a documented quality assurance program that has been determined to me...

--- System-level requirements ---
  1. [system] Provide a dry storage preparation area where spent fuel (in the transfer cask) is prepared for loading into dry storage ...
  2. [system] Provide a dry storage preparation area where spent fuel (in the transfer cask) is prepared for loading into dry storage ...
  3. [system] All Factory Acceptance Test Procedures and Site Acceptance Procedures shall be completed by the Supplier and submitted t...
  4. [system] All Factory Acceptance Test Procedures and Site Acceptance Procedures shall be completed by the Supplier and submitted t...


Optional: Nova summary

In [ ]:
from config import NOVA_MODEL_ID
from src.nova_tier import _nova_invoke, get_bedrock_client

def _build_context(nodes):
    return "\n\n".join(f"[{tier}] {text}" for _, tier, text in nodes if text)

all_nodes = ([plant_node] if plant_node else []) + systems + components
context = _build_context(all_nodes)
question = "Summarize the system-level requirements and how they relate to the plant."
client = get_bedrock_client()
prompt = f"""Use ONLY the following requirements to answer the question.

Requirements:
{context[:28000]}

Question: {question}

Answer:"""
answer = _nova_invoke(client, NOVA_MODEL_ID, prompt, max_tokens=1024)
print(answer)

## 2. Free-text question

In [4]:
def _cosine_sim(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb + 1e-12)

def get_seed_ids(query_embedding, full_ids, embeddings, top_k=5):
    scored = [(i, _cosine_sim(query_embedding, emb)) for i, emb in enumerate(embeddings)]
    scored.sort(key=lambda x: -x[1])
    return [full_ids[i] for i, _ in scored[:top_k]]

def get_subgraph(seed_ids, driver, max_hops=2):
    with driver.session() as session:
        result = session.run(
            """
            MATCH (start:Requirement) WHERE start.id IN $ids
            OPTIONAL MATCH (start)-[:TRACES_TO*1..""" + str(max_hops) + """]-(n:Requirement)
            WITH collect(DISTINCT start) + collect(DISTINCT n) AS allNodes
            UNWIND allNodes AS node
            WITH DISTINCT node WHERE node IS NOT NULL
            RETURN node.id AS id, node.tier AS tier, node.text AS text
            """,
            ids=seed_ids,
        )
        return [(r["id"], r["tier"] or "", (r["text"] or "")[:500]) for r in result]

def build_context(subgraph_nodes):
    lines = []
    for req_id, tier, text in subgraph_nodes:
        lines.append(f"[{tier}] {text}")
    return "\n\n".join(lines)

def ask_nova(question, context, client=None):
    from config import NOVA_MODEL_ID
    from src.nova_tier import _nova_invoke, get_bedrock_client
    if client is None:
        client = get_bedrock_client()
    prompt = f"""You are a requirements engineer. Use ONLY the following requirements (from a traceability graph) to answer the question. If the context does not contain enough information, say so.

Requirements (tier and text):
{context}

Question: {question}

Answer:"""
    return _nova_invoke(client, NOVA_MODEL_ID, prompt[:30000], max_tokens=1024)

Run

In [5]:
from src.titan_embeddings import embed_text, get_bedrock_client
from src.neo4j_loader import get_driver

QUESTION = "What requirements relate to pressurizer or RCP?"
TOP_K_SEEDS = 5
MAX_HOPS = 2

client = get_bedrock_client()
query_embedding = embed_text(QUESTION, client=client)
if not query_embedding:
    print("Failed to embed question.")
else:
    seed_ids = get_seed_ids(query_embedding, full_ids, embeddings, top_k=TOP_K_SEEDS)
    driver = get_driver()
    subgraph = get_subgraph(seed_ids, driver, max_hops=MAX_HOPS)
    driver.close()
    context = build_context(subgraph)
    print(f"Seeds: {len(seed_ids)}, subgraph nodes: {len(subgraph)}")
    answer = ask_nova(QUESTION, context, client=client)
    print("---")
    print(answer)

Seeds: 5, subgraph nodes: 52
---
Based on the provided requirements, the following requirements relate to the pressurizer or RCP:

1. **[component] The RCP is a vertical, centrifugal type pump. The pump is designed to use traditional seals designed to allow continued operation of the pump for a period of at least one week after failure of one of the several stages in order to permit an orderly plant shutdown, without excessive loss of reactor coolant, in the event of the degradation of a second seal stage. The RCP has provisions to prevent motor damage due to overheating by the use of an onboard heat exchanger that is supplied with component cooling water. Two 50 percent capacity pumps are provided, one on each cold leg.**

2. **[component] The RCP has provisions to prevent motor damage due to overheating by the use of an onboard heat exchanger that is supplied with component cooling water. Two 50 percent capacity pumps are provided, one on each cold leg.**

3. **[component] 1. The Rea

Another question

In [ ]:
def graph_rag(question, top_k=5, max_hops=2):
    client = get_bedrock_client()
    query_embedding = embed_text(question, client=client)
    if not query_embedding:
        return "Failed to embed question."
    seed_ids = get_seed_ids(query_embedding, full_ids, embeddings, top_k=top_k)
    driver = get_driver()
    subgraph = get_subgraph(seed_ids, driver, max_hops=max_hops)
    driver.close()
    context = build_context(subgraph)
    return ask_nova(question, context, client=client)

# Uncomment and run:
# print(graph_rag("Which system requirements address safety valves?"))